# Advanced Quantum Error Mitigation: DD, Pauli Twirling, PEC, & Symmetry Verification

In this notebook, we implement and simulate four advanced error mitigation strategies designed to cover idle qubit dephasing, coherent error twirling, sampling-based cancellation, and subspace leakage post-selection.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quantum_mitigation import (
    noisy_simulator, ideal_simulator,
    twirled_cz_circuit, TWIRL_PAIRS,
    run_pec_simulation, run_parity_verified_grover,
    plot_comparison_bar
)
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
%matplotlib inline


## 1. Dynamical Decoupling (Spin Echo)

We simulate active spin echo ($X - \text{delay} - X$) refocusing to counteract systematic environmental phase detuning ($Z$-drift rotations) on idle qubits.


In [ ]:
def simulate_dephasing_and_dd():
    sim = AerSimulator()
    drift_angle = np.pi / 4
    
    # 1. Unmitigated (Accumulates drift)
    qc_unmit = QuantumCircuit(1, 1)
    qc_unmit.h(0)
    qc_unmit.rz(drift_angle, 0)
    qc_unmit.rz(drift_angle, 0)
    qc_unmit.h(0)
    qc_unmit.measure(0, 0)
    
    # 2. Mitigated (CPMG decoupling)
    qc_mit = QuantumCircuit(1, 1)
    qc_mit.h(0)
    qc_mit.rz(drift_angle, 0)
    qc_mit.x(0)
    qc_mit.rz(drift_angle, 0)
    qc_mit.x(0)
    qc_mit.h(0)
    qc_mit.measure(0, 0)
    
    shots = 10000
    c_unmit = sim.run(transpile(qc_unmit, sim), shots=shots).result().get_counts()
    c_mit = sim.run(transpile(qc_mit, sim), shots=shots).result().get_counts()
    
    print("Dynamical Decoupling (Spin Echo) Results:")
    print(f"  Unmitigated P(|0⟩) = {c_unmit.get('0', 0)/shots:.4f} (Ideal = 1.0000)")
    print(f"  Mitigated P(|0⟩)   = {c_mit.get('0', 0)/shots:.4f} (Ideal = 1.0000)")
    
    print("\nUnmitigated Circuit Layout:")
    print(qc_unmit.draw('text'))
    print("\nMitigated (DD) Circuit Layout:")
    print(qc_mit.draw('text'))

simulate_dephasing_and_dd()


## 2. Randomized Compiling (Pauli Twirling)

Randomized Compiling inserts matching random single-qubit Pauli operations before and after entangling CZ gates, twirling coherent over-rotation errors into simple stochastic Pauli channel decay.


In [ ]:
sim = AerSimulator()
shots = 8000
theta_err = 0.25

# 1. No twirling
qc_no_twirl = twirled_cz_circuit(theta=theta_err, twirl_pair=None)
c_no_twirl = sim.run(transpile(qc_no_twirl, sim), shots=shots).result().get_counts()
p_00_no_twirl = c_no_twirl.get('00', 0) / shots

# 2. With RC
p_list = []
for pre in list(TWIRL_PAIRS.keys()):
    post = TWIRL_PAIRS[pre]
    qc_t = twirled_cz_circuit(theta=theta_err, twirl_pair=(pre, post))
    c = sim.run(transpile(qc_t, sim), shots=shots).result().get_counts()
    p_list.append(c.get('00', 0) / shots)
p_00_twirled = np.mean(p_list)

print(f"Ideal success probability P(|00⟩) = 1.0000")
print(f"Coherent error (no twirling) P(|00⟩)    = {p_00_no_twirl:.4f}  (error = {abs(1.0 - p_00_no_twirl):.4f})")
print(f"Twirled error (RC averaged) P(|00⟩)     = {p_00_twirled:.4f}  (error = {abs(1.0 - p_00_twirled):.4f})")


In [ ]:
fold_factors = [1, 2, 3, 4, 5]
coh_probs = []
tw_probs = []

for ff in fold_factors:
    t_scaled = theta_err * ff
    
    # Coherent
    qc = twirled_cz_circuit(theta=t_scaled, twirl_pair=None)
    coh_probs.append(sim.run(transpile(qc, sim), shots=4000).result().get_counts().get('00', 0) / 4000)
    
    # Twirled
    p_l = []
    for pre in list(TWIRL_PAIRS.keys())[:5]:
        post = TWIRL_PAIRS[pre]
        qc_t = twirled_cz_circuit(theta=t_scaled, twirl_pair=(pre, post))
        p_l.append(sim.run(transpile(qc_t, sim), shots=2000).result().get_counts().get('00', 0) / 2000)
    tw_probs.append(np.mean(p_l))

plt.figure(figsize=(8, 5))
plt.plot(fold_factors, coh_probs, 'o-', color='#ef4444', label='Coherent Error (non-twirled)', linewidth=2)
plt.plot(fold_factors, tw_probs, 's--', color='#10b981', label='Twirled Error (RC)', linewidth=2)
plt.axhline(y=1.0, color='gray', linestyle=':')
plt.xlabel('Noise Scaling Factor (Fold Factor)', fontsize=12)
plt.ylabel('Success Probability P(|00⟩)', fontsize=12)
plt.title('Randomized Compiling: Converting Coherent to Stochastic Noise', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 3. Probabilistic Error Cancellation (PEC)

PEC uses quasi-probability sampling to express exact inverse noise maps, canceling single-qubit phase flips at the cost of statistical sampling overhead $\gamma$.


In [ ]:
mit_exp, raw_exp, gamma = run_pec_simulation(p_noise=0.10, shots=500)

print("Probabilistic Error Cancellation (PEC) Results:")
print(f"  Ideal Expectation <X>       = 1.0000")
print(f"  Unmitigated Expectation <X> = {raw_exp:.4f} (error = {abs(1.0 - raw_exp):.4f})")
print(f"  PEC Mitigated Expectation <X> = {mit_exp:.4f} (error = {abs(1.0 - mit_exp):.4f})")
print(f"  Sampling cost overhead (γ)   = {gamma:.4f}")


## 4. Symmetry Verification & Post-Selection

We append an ancillary check qubit to projectively measure the state parity at measurement time, discarding trials that fail our parity symmetry invariant under bit-flip noise.


In [ ]:
unmit_s, mit_s, discard_r = run_parity_verified_grover(p_bit_flip=0.08, shots=15000)

print("Symmetry Verification & Post-Selection Results:")
print(f"  Ideal P(|11⟩)          = 1.0000")
print(f"  Unmitigated P(|11⟩)     = {unmit_s:.4f}  (error = {abs(1.0 - unmit_s):.4f})")
print(f"  Post-Selected P(|11⟩)   = {mit_s:.4f}  (error = {abs(1.0 - mit_s):.4f})")
print(f"  Discarded Noisy Shots  = {discard_r * 100:.1f}%")

# Plot comparison
plot_comparison_bar(['Unmitigated', 'Symmetry Verified', 'Ideal'], [unmit_s, mit_s, 1.0], title="Grover's Search: Post-Selection via Parity Check")
plt.show()
